<a href="https://colab.research.google.com/github/rinutitus/Lora-finetuning-Llama-3---Parenting-dataset/blob/main/Train_ollama_with_parenting_data_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Below line test if connected to GPU**

In [ ]:
import torch
torch.cuda.is_available()


True

In [ ]:
!pip install -q unsloth datasets transformers accelerate bitsandbytes huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

1. Above I am installing depndencies


2. below :Login to Hugging Face

In [ ]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


3️⃣ Upload Your Dataset

Upload the JSON dataset you created earlier. --This is to upload to google collab , Optional since I have already uploaded data set to hugging face

In [ ]:
from google.colab import files
files.upload()

Saving parenting_dataset_250.json to parenting_dataset_250.json


{'parenting_dataset_250.json': b'[\n  {\n    "instruction": "Provide parenting guidance",\n    "input": "My 10-year-old is refusing to eat food.",\n    "output": "Keep consequences logical and calm. Stay calm and validate feelings. Model the behavior you expect. Praise effort instead of outcome."\n  },\n  {\n    "instruction": "Provide parenting guidance",\n    "input": "My 7-year-old is saying \'I am not good at anything\'.",\n    "output": "Keep consequences logical and calm. Schedule focused one-on-one time. Use short breaks and return to task. Reduce distractions during conversations."\n  },\n  {\n    "instruction": "Provide parenting guidance",\n    "input": "My 5-year-old is fear of sleeping alone.",\n    "output": "Avoid shaming and focus on guidance. Praise effort instead of outcome. Use short breaks and return to task. Reduce distractions during conversations."\n  },\n  {\n    "instruction": "Provide parenting guidance",\n    "input": "My 7-year-old is not listening to instruc

4️⃣ Load Dataset

I am going to load data set from hugging face

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rinutitus/parenting-ai-dataset")
dataset = dataset["train"]

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/342 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.86k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 250
})

In [ ]:
print(dataset.column_names)

['instruction', 'input', 'output']


5️⃣ Format Dataset for Instruction Training -- to the format intsruction,input,output if already not in this format

In [ ]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_prompt)


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
6️⃣ Load Llama-3 with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "meta-llama/Meta-Llama-3-8B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


7️⃣ Apply LoRA (Efficient Training)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
)


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


8️⃣ Tokenize Dataset

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

dataset = dataset.map(tokenize_function)


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

9️⃣ Train Model

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    tokenizer = tokenizer,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, #4*max_steps it will go through data
        warmup_steps = 10,
        max_steps = 200, #go through data set 200 times
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 10,
        output_dir = "outputs",
    ),
)

trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 7 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
10,3.466513
20,1.636779
30,0.676586
40,0.533249
50,0.481786
60,0.415881
70,0.351816
80,0.342120
90,0.334888
100,0.330067


TrainOutput(global_step=200, training_loss=0.5877182579040527, metrics={'train_runtime': 454.2091, 'train_samples_per_second': 3.523, 'train_steps_per_second': 0.44, 'total_flos': 3718076392243200.0, 'train_loss': 0.5877182579040527, 'epoch': 6.256})

🔟 Save the Fine-Tuned Model

In [ ]:
model.save_pretrained("parenting_llama3_lora")
tokenizer.save_pretrained("parenting_llama3_lora")


('parenting_llama3_lora/tokenizer_config.json',
 'parenting_llama3_lora/tokenizer.json')

1️⃣1️⃣ Download the Model

In [ ]:
from google.colab import files
!zip -r parenting_llama3_lora.zip parenting_llama3_lora
files.download("parenting_llama3_lora.zip")


  adding: parenting_llama3_lora/ (stored 0%)
  adding: parenting_llama3_lora/tokenizer.json (deflated 85%)
  adding: parenting_llama3_lora/adapter_model.safetensors (deflated 8%)
  adding: parenting_llama3_lora/tokenizer_config.json (deflated 45%)
  adding: parenting_llama3_lora/adapter_config.json (deflated 57%)
  adding: parenting_llama3_lora/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1️⃣2️⃣ Test the Model

In [ ]:
prompt = """
### Instruction:
Give advice to handle toddler tantrum.

### Input:
My 2 year old screams in the supermarket.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(outputs[0]))


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

<|begin_of_text|>
### Instruction:
Give advice to handle toddler tantrum.

### Input:
My 2 year old screams in the supermarket.

### Response:
Keep consequences logical and calm. Schedule focused one-on-one time. Offer limited choices to encourage cooperation. Avoid shaming and focus on guidance. Encourage problem-solving discussion. Praise effort instead of outcome. Reinforce positive behavior immediately. Teach simple coping skills like deep breathing. Maintain eye contact and speak gently. Use short breaks and return to task. Set clear and consistent boundaries. Stay calm and validate feelings. Encourage creativity and offer choices. Schedule focused one-on-one time. Reduce distractions during conversations. Maintain eye contact and speak gently. Teach simple coping skills like deep breathing. Use short breaks and return to task. Avoid comparisons with others. Set clear and consistent expectations. Reinforce positive behavior immediately. Stay calm and model the behavior you expect.